# Qlora and WandB : 
&nbsp;&nbsp;&nbsp; In this notebook, we will try to combine both SRT using Qlora and we will add a WandB component to track the metrics from it.

## Requirements Installation : 
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will try to install the required libraries needed for the overall process. 

In [1]:
!pip install unsloth
!pip install --upgrade --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install trl peft accelerate bitsandbytes datasets transformers

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sp3goznm/unsloth_edee8608648f45ef9ebdca47f2f88298
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sp3goznm/unsloth_edee8608648f45ef9ebdca47f2f88298
  Resolved https://github.com/unslothai/unsloth.git to commit b09aa82a3ac9ac9794c497e4b8f747f77e52b162
  Installing build dependencies ... done
  Getting requirements to build wheel ... one
  Preparing metadata (pyproject.toml) ... one


In [2]:
! conda install -y gcc -c conda-forge

Channels:
 - conda-forge
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: | 


==> WARNING: A newer version of conda exists. <==
    current version: 25.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [3]:
! conda install -c nvidia cuda-toolkit

Channels:
 - nvidia
 - conda-forge
Platform: linux-64
doneecting package metadata (repodata.json): - 
doneing environment: | 


==> WARNING: A newer version of conda exists. <==
    current version: 25.1.1
    latest version: 26.3.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



# All requested packages already installed.



In [4]:
!pip uninstall wandb -y
!pip cache purge
!rm -rf /opt/conda/lib/python3.11/site-packages/wandb
!rm -rf /opt/conda/lib/python3.11/site-packages/wandb-*.dist-info
!pip install --no-cache-dir wandb

Found existing installation: wandb 0.19.1
Uninstalling wandb-0.19.1:
  Successfully uninstalled wandb-0.19.1
Files removed: 18 (1.6 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 6.5 MB/s eta 0:00:00a 0:00:01


In [ ]:
!pip install --no-cache-dir "wandb==0.19.1"

   ━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/20.0 MB 19.1 MB/s eta 0:00:01

## Setting WandB :
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will try to setup the weight and biases so that it trackes and plots relevant to the performance of our model. 

In [1]:
import wandb
wandb.login()
wandb.init(
    project="cuda-kernel-optimization",
    name="cuda_kernels_sft",
    notes="SFT for Cuda kernels code generation"
)

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: khadidja-bahfir (khadidja-bahfir-other). Use `wandb login --relogin` to force relogin


## Setting Model : 
&nbsp;&nbps;&nbsp; In this section of the notebook, we will try to set the model. 

In [2]:
from unsloth import FastLanguageModel
import torch
import os 
max_seq_length = 5100  
dtype = None           
load_in_4bit = True  # Qlora uses the quantized model rather than the original base model    
os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Qwen2.5-Coder-7B-Instruct",
    max_seq_length=max_seq_length,
    dtype = dtype, 
    load_in_4bit=True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA H100 NVL MIG 1g.24gb. Num GPUs = 1. Max memory: 21.625 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 9.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [3]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                         
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = 16,
    lora_dropout = 0,               
    bias = "none",               
    use_gradient_checkpointing = "unsloth",  
    random_state = 42,
    use_rslora = False,
    loftq_config = None,
)

model.print_trainable_parameters()

Unsloth 2026.4.8 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


trainable params: 40,370,176 || all params: 7,655,986,688 || trainable%: 0.5273


## Loading And Setting The Dataset : 
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will go ahead and set the dataset along with crafting a prompt that is aligned to convey the overall instructions required by our LLM. 

In [4]:
from datasets import load_dataset, concatenate_datasets

dataset = load_dataset("SakanaAI/AI-CUDA-Engineer-Archive" , split = "level_1" )
print(f" We have {len(dataset)} samples")

 We have 12157 samples


In [5]:
split_dataset = dataset.train_test_split(test_size=0.02, seed=42)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]

print(f"Train size: {len(train_dataset)}")
print(f"Test size: {len(test_dataset)}")

Train size: 11913
Test size: 244


## Prompt Refinement : 
&nbsp;&nbsp;&nbsp; In this section of the notebook, we will try to craft a prompt for our SFT part of the whole process, we will try to go ahead and use the autokernel playbook that included certain pieces of information to follow in the optimization of the kernel, as our approach differ from the autokernel approach we will just take the portion of the 6 tier and remove the details that they have used to convey their overall pipeline.  <br> 
https://github.com/RightNow-AI/autokernel/blob/main/program.md#cuda-c-optimization-playbook

In [6]:
cuda_rl_task = """
You are an expert CUDA kernel optimizer. Your output must be ONLY a single ```cuda ... ``` block. No explanations, no markdown text, no tables, no planning — ONLY the code block.

The code must be a COMPLETE, standalone CUDA C++ file that compiles with: nvcc file.cu -o file

## STRICT RULES
1. NO #include <torch/extension.h> or any PyTorch headers.
2. ONLY use <cuda_runtime.h>, <stdio.h>, <math.h>.
3. Every #define, macro, and constant used MUST be defined at the top of the file before use.
4. File must contain:
   - A function named baseline_kernel() — copy of the original kernel, unmodified.
   - A function named optimized_kernel() — your optimized version.
   - A main() function that:
       * cudaMalloc all buffers
       * Initializes inputs on host, copies with cudaMemcpy
       * Launches baseline_kernel then optimized_kernel
       * Calls cudaDeviceSynchronize() after each launch
       * Compares outputs element-wise and prints MATH_FAILED if they differ
       * cudaFree all buffers
5. Use fixed small sizes (e.g. 128x128) to avoid memory issues.
6. Annotate each optimization inline: // OPT[T2]: float4 loads -> coalesced 128-bit access
7. Output NOTHING except the ```cuda ... ``` block.

## KERNEL TO OPTIMIZE
Operation: {Op_Name}
```cuda
{CUDA_Code}
```

## OPTIMIZATION TIERS (apply in order, annotate each with // OPT[Tn])
Tier 1 - Thread/Block Config (10-50% gain)
- Sweep blockDim.x: 128, 256, 512; default 256
- Add __launch_bounds__(maxThreadsPerBlock, minBlocksPerMultiprocessor)

Tier 2 - Memory Access (20-40% gain)
- Tile global loads into __shared__ memory
- Pad shared arrays to avoid bank conflicts: [32][33] not [32][32]
- Use float4 for 128-bit loads; __ldg() for read-only pointers
- Ensure warp threads access consecutive addresses

Tier 3 - Tensor Cores for matmul-like ops (2-4x gain)
- wmma API: 16x16x16 (fp16) or 8x8x4 (tf32) tiles
- Accumulate in fp32; convert to fp16 only at epilogue

Tier 4 - Advanced Pipelining (10-20% gain)
- Persistent kernels: launch SM_COUNT blocks, loop over tiles inside
- cp.async (Ampere+) for async global->shared prefetch

Tier 5 - Op-Specific Patterns
- MatMul      : swizzled shmem, register-level 4x4 tiling, fused epilogue
- Softmax     : __shfl_xor_sync reduction, __expf(), online max subtraction
- LayerNorm   : Welford's algorithm, __shfl_down_sync, rsqrtf()
- Cross Entropy: fused log-sum-exp, warp reductions
- RoPE        : __sincosf(), constant-memory freq table, half2 ops

## ANTI-PATTERNS TO ELIMINATE
- Warp divergence in inner loops      -> hoist branches outside
- Uncoalesced memory access           -> consecutive thread->address mapping
- Shared memory bank conflicts        -> pad or swizzle layout
- Register spilling                   -> use __launch_bounds__
- Excess __syncthreads__()            -> minimize sync points
- Global atomics in hot paths         -> warp-reduce first, one atomic per warp
- Missing __restrict__ on read-only pointers
"""

In [7]:

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    texts = []
    for op, cuda_code in zip(examples["Op_Name"], examples["CUDA_Code"]):
        text = cuda_rl_task.format(
            Op_Name   = op,
            CUDA_Code = cuda_code,
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)
print(f"Total training samples: {len(train_dataset)}")
print (f"Total testing examples : {len(test_dataset)}")
print(train_dataset[0]["text"])

Map:   0%|          | 0/11913 [00:00<?, ? examples/s]

Map:   0%|          | 0/244 [00:00<?, ? examples/s]

Total training samples: 11913
Total testing examples : 244

You are an expert CUDA kernel optimizer. Your output must be ONLY a single ```cuda ... ``` block. No explanations, no markdown text, no tables, no planning — ONLY the code block.

The code must be a COMPLETE, standalone CUDA C++ file that compiles with: nvcc file.cu -o file

## STRICT RULES
1. NO #include <torch/extension.h> or any PyTorch headers.
2. ONLY use <cuda_runtime.h>, <stdio.h>, <math.h>.
3. Every #define, macro, and constant used MUST be defined at the top of the file before use.
4. File must contain:
   - A function named baseline_kernel() — copy of the original kernel, unmodified.
   - A function named optimized_kernel() — your optimized version.
   - A main() function that:
       * cudaMalloc all buffers
       * Initializes inputs on host, copies with cudaMemcpy
       * Launches baseline_kernel then optimized_kernel
       * Calls cudaDeviceSynchronize() after each launch
       * Compares outputs element-wise

In [8]:
tokens = tokenizer(train_dataset[0]["text"], return_tensors="pt")
print(f"Token count: {tokens['input_ids'].shape[1]}")

Token count: 2232


## SFT : 
&nbsp;&nbsp;&nbsp; In this section of the pipeline, we are set to apply the SFT component to get the qwen-code 7b to be familiar with CUDA, so some minor adjustments have been carried out as follows : 
- We will start by testing the current performance of the qwen-code on CUDA code generation.
- We will then fine-tune the LLM on CUDA kernels using a simple training signal while monitoring the overall improvement using a minor reward that assigns -1 for non-executable and non-correct and 1 for both and 0 for either one failing. 
- We will compare the results before and after the finetuning to testify for how well the SFT component helps in getting the LLM familiar with CUDA code.


In [9]:
import tempfile, os, textwrap

In [10]:
def extract_cuda_code(response: str) -> str | None:
    pattern = r"```(?:cuda|cpp|c\+\+)?\s*([\s\S]*?)```"
    matches = re.findall(pattern, response, re.IGNORECASE)
    for m in matches:
        if any(kw in m for kw in ["__global__", "cudaMalloc", "blockIdx", "threadIdx", "PYBIND11_MODULE"]):
            return m.strip()
    return None

In [11]:
def run_evaluation(model, tokenizer, subset, tag: str = "") -> list[dict]:
    records = []
    for sample in tqdm(subset, desc=f"Evaluating [{tag}]"):
        op_name   = sample.get("Op_Name")   or "unknown_op"
        cuda_code = sample.get("CUDA_Code") or ""

        # Skip samples with no code at all
        if not cuda_code.strip():
            records.append(dict(op=op_name, executable=False,
                                correct=False, reward=-1,
                                error="Empty CUDA_Code field"))
            continue

        response  = generate_kernel(model, tokenizer, op_name, cuda_code)
        extracted = extract_cuda_code(response)

        if extracted is None:
            records.append(dict(op=op_name, executable=False,
                                correct=False, reward=-1,
                                error="No CUDA block found in response"))
            continue

        result = evaluate_kernel(extracted, op_name)
        result["op"] = op_name
        records.append(result)

    return records

### Prompt : 
As in this part of the pipeline, we are trying to only get the LLM to be familiar with the CUDA kernel, and we are not going to measure the optimization overall. We will experiment with different prompts. 

In [12]:
def generate_kernel(model, tokenizer, op_name: str, cuda_code, 
                    max_new_tokens: int = 2486) -> str:
    FastLanguageModel.for_inference(model)

    # Guard: ensure both inputs are clean strings
    if not isinstance(cuda_code, str) or not cuda_code.strip():
        return ""
    if not isinstance(op_name, str) or not op_name.strip():
        op_name = "unknown_op"

    # Truncate cuda_code tokens before building the prompt
    code_tokens = tokenizer.encode(cuda_code, add_special_tokens=False)
    if len(code_tokens) > 800:
        cuda_code = tokenizer.decode(code_tokens[:800]) + "\n// ... (truncated)"

    messages = [
    {
        "role": "system",
        "content": cuda_rl_task.format(Op_Name=op_name, CUDA_Code=cuda_code),
    },
    {
        "role": "user",
        "content": (
            f"Optimize this CUDA kernel for operation: {op_name}\n\n"
            f"```cuda\n{cuda_code}\n```"
        ),
    },
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            top_p=0.95,
            repetition_penalty=1.1,
            use_cache=True,
        )

    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)

In [13]:
import os
import json
import time
import uuid

PENDING_DIR = "pending_kernels"
COMPLETED_DIR = "completed_results"
POLL_INTERVAL = 0.5
POLL_TIMEOUT = 60.0  # max seconds to wait for Script 2 to finish

def evaluate_kernel(cuda_code: str, op_name: str) -> dict:
    """
    Writes the kernel as a harness .cu file into pending_kernels/,
    waits for Script 2 to produce a result in completed_results/,
    and returns the result dict with reward, status, and metrics.
    """
    trial_id = f"{op_name}_{uuid.uuid4().hex[:8]}"
    harness_path = os.path.join(PENDING_DIR, f"{trial_id}_harness.cu")
    result_path  = os.path.join(COMPLETED_DIR, f"{trial_id}_results.json")

    os.makedirs(PENDING_DIR, exist_ok=True)
    os.makedirs(COMPLETED_DIR, exist_ok=True)

    # Drop the .cu file for Script 2 to pick up
    with open(harness_path, "w") as f:
        f.write(cuda_code)

    # Poll until Script 2 writes the result
    deadline = time.time() + POLL_TIMEOUT
    while time.time() < deadline:
        if os.path.exists(result_path):
            with open(result_path, "r") as f:
                result = json.load(f)
            return {
                "op":        result.get("trial_id"),
                "executable": result.get("status") not in ("COMPILATION_ERROR", "TIMEOUT", "RUNTIME_ERROR"),
                "correct":   result.get("status") == "SUCCESS",
                "reward":    result.get("reward", -1.0),
                "status":    result.get("status"),
                "metrics":   result.get("metrics", {}),
                "error":     result.get("error_trace"),
            }
        time.sleep(POLL_INTERVAL)

    # Timed out waiting
    return {
        "op":        op_name,
        "executable": False,
        "correct":   False,
        "reward":    -1.0,
        "status":    "POLL_TIMEOUT",
        "metrics":   {},
        "error":     f"No result after {POLL_TIMEOUT}s — Script 2 may not be running.",
    }

In [17]:
import random
from tqdm import tqdm
import re 
import warnings
from torch.utils.cpp_extension import load_inline
warnings.filterwarnings("ignore")


N_EVAL = 8 
random.seed(42)
eval_indices = random.sample(range(len(test_dataset)), min(N_EVAL, len(test_dataset)))
eval_subset  = test_dataset.select(eval_indices)


def run_evaluation(model, tokenizer, subset, tag: str = "") -> list[dict]:
    records = []
    for sample in tqdm(subset, desc=f"Evaluating [{tag}]"):
        op_name   = sample.get("Op_Name")   or "unknown_op"
        cuda_code = sample.get("CUDA_Code") or ""

        if not cuda_code.strip():
            records.append(dict(op=op_name, executable=False,
                                correct=False, reward=-1,
                                error="Empty CUDA_Code field"))
            continue

        response  = generate_kernel(model, tokenizer, op_name, cuda_code)
        print(f"The response generated is : {response}") 
        extracted = extract_cuda_code(response)

        if extracted is None:
            records.append(dict(op=op_name, executable=False,
                                correct=False, reward=-1,
                                error="No CUDA block found in response"))
            continue

        # extracted must be a full harness .cu (baseline + optimized)
        # that Script 2 can compile and profile directly
        result = evaluate_kernel(extracted, op_name)
        print (f"The result is {result}")
        records.append(result)

    return records


def summarise(records: list[dict], label: str):
    total      = len(records)
    exec_count = sum(r["executable"] for r in records)
    corr_count = sum(r["correct"]    for r in records)
    avg_reward = sum(r["reward"]     for r in records) / total

    print(f"\n{'='*55}")
    print(f" {label}")
    print(f"{'='*55}")
    print(f" Total evaluated  : {total}")
    print(f" Executable       : {exec_count:>3}  ({100*exec_count/total:.1f}%)")
    print(f" Correct          : {corr_count:>3}  ({100*corr_count/total:.1f}%)")
    print(f" Average reward   : {avg_reward:+.3f}")
    print(f"{'='*55}\n")
    return dict(total=total, exec_rate=exec_count/total,
                corr_rate=corr_count/total, avg_reward=avg_reward)


pre_records  = run_evaluation(model, tokenizer, eval_subset, tag="PRE")
pre_metrics  = summarise(pre_records, "PRE-FINE-TUNING RESULTS")

Both `max_new_tokens` (=2486) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The response generated is : ```cuda
#include <cuda_runtime.h>
#include <stdio.h>
#include <math.h>

#define BLOCK_SIZE 256
#define TILE_DIM (BLOCK_SIZE * 2)
#define CUBLAS_THRESHOLD 512

__constant__ float __A[TILE_DIM * TILE_DIM];
__constant__ float __B[TILE_DIM * TILE_DIM];

__global__ void baseline_kernel(float* __restrict__ C, int M, int K, int N) {
    int row0 = blockIdx.y * TILE_DIM + threadIdx.y;
    int col0 = blockIdx.x * TILE_DIM + threadIdx.x;
    int row1 = row0 + BLOCK_SIZE;
    int col1 = col0 + BLOCK_SIZE;

    float Cvalue00 = 0.0f, Cvalue01 = 0.0f, Cvalue10 = 0.0f, Cvalue11 = 0.0f;

    __shared__ float As[TILE_DIM][TILE_DIM];
    __shared__ float Bs[TILE_DIM][TILE_DIM];

    for (int tile = 0; tile < (K + TILE_DIM - 1) / TILE_DIM; tile++) {
        int tStart = tile * TILE_DIM;

        #pragma unroll
        for (int i = 0; i < 2; i++) {
            int aRow = blockIdx.y * TILE_DIM + threadIdx.y + i * BLOCK_SIZE;
            for (int j = 0; j < 2; j++) {
           

Both `max_new_tokens` (=2486) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The result is {'op': '8_Matmul_with_irregular_shapes__139d901f', 'executable': False, 'correct': False, 'reward': -1.0, 'status': 'COMPILATION_ERROR', 'metrics': {'baseline_time_ns': None, 'optimized_time_ns': None, 'occupancy_pct': None, 'compute_pct': None, 'memory_pct': None}, 'error': "nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).\nptxas error   : Entry function '_Z16optimized_kernelPfiii' uses too much shared data (0x200000 bytes, 0xc000 max)\nptxas error   : File uses too much global constant data (0x200000 bytes, 0x10000 max)\nptxas error   : Entry function '_Z15baseline_kernelPfiii' uses too much shared data (0x200000 bytes, 0xc000 max)"}
The response generated is : ```cuda
#include <cuda_runtime.h>
#include <stdio.h>
#include <math.h>

#define TILE_K 32
#define BLOCK_SIZE 256

__global__ void baseline_kernel(
    const float* __restrict__

Both `max_new_tokens` (=2486) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

The response generated is : ```cuda
#define BLOCK_SIZE 256
#define TILE_WIDTH 16

template <typename scalar_t>
__global__ void baseline_kernel(
    const scalar_t* __restrict__ input,
    const scalar_t* __restrict__ weight,
    const scalar_t* __restrict__ bias,
    scalar_t* __restrict__ output,
    const int batch_size,
    const int in_channels,
    const int in_height,
    const int in_width,
    const int out_channels,
    const int kernel_h,
    const int kernel_w,
    const int stride,
    const int padding,
    const int output_padding,
    const int groups,
    const int dilation,
    const int out_height,
    const int out_width
) {
    const int idx = blockIdx.x * blockDim.x + threadIdx.x;
    const int total_elements = batch_size * out_channels * out_height * out_width;
    if (idx >= total_elements) return;

    // Unravel output index
    const int ow = idx % out_width;
    const int oh = (idx / out_width) % out_height;
    const int oc = (idx / (out_width * out_height))

Both `max_new_tokens` (=2486) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The result is {'op': '98_KLDivLoss_35b4c5d0', 'executable': False, 'correct': False, 'reward': -1.0, 'status': 'COMPILATION_ERROR', 'metrics': {'baseline_time_ns': None, 'optimized_time_ns': None, 'occupancy_pct': None, 'compute_pct': None, 'memory_pct': None}, 'error': "nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).\n/opt/conda/bin/../lib/gcc/x86_64-conda-linux-gnu/14.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /tmp/tmpxft_00002e2c_00000000-11_98_KLDivLoss_35b4c5d0_harness.o: warning: relocation against `__device_builtin_variable_gridDim' in read-only section `.text'\n/opt/conda/bin/../lib/gcc/x86_64-conda-linux-gnu/14.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: /tmp/tmpxft_00002e2c_00000000-11_98_KLDivLoss_35b4c5d0_harness.o: in function `main':\ntmpxft_00002e2c_00000000-6_98_KLDivLoss_35b4c5d0_harness.cudafe1.cpp:(.text+0x5b): undefined re

Evaluating [PRE]: 100%|██████████| 8/8 [11:20<00:00, 85.11s/it] 

The result is {'op': '74_conv_transposed_1D_dilated_9bef9a63', 'executable': False, 'correct': False, 'reward': -1.0, 'status': 'COMPILATION_ERROR', 'metrics': {'baseline_time_ns': None, 'optimized_time_ns': None, 'occupancy_pct': None, 'compute_pct': None, 'memory_pct': None}, 'error': 'nvcc warning : Support for offline compilation for architectures prior to \'<compute/sm/lto>_75\' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).\npending_kernels/74_conv_transposed_1D_dilated_9bef9a63_harness.cu(34): error: a variable length array cannot have static storage duration\n      __attribute__((shared)) float s_weight[C_in * C_out * K_w];\n                                    ^\n\npending_kernels/74_conv_transposed_1D_dilated_9bef9a63_harness.cu(90): error: identifier "total_elements" is undefined\n      baseline_kernel<<<(total_elements + 256 - 1) / 256, 256>>>(d_x, d_weight, d_bias, d_y, N, C_in, C_out, L_in, L_out, K_w, stride, padding, dilation);

## SFT Component : 
&nsbp;&nbsp;&nbsp; here, we will have an SFT component that is tasked with predicting the next token and we will see how it overall improves the familiarity with cuda code. 

In [18]:
from transformers import TrainerCallback

class RewardLoggingCallback(TrainerCallback):
    def __init__(self, model, tokenizer, eval_subset, every_n_steps=50):
        self.model = model
        self.tokenizer = tokenizer
        self.eval_subset = eval_subset 
        self.every_n_steps = every_n_steps

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % self.every_n_steps != 0:
            return
        records = run_evaluation(self.model, self.tokenizer, self.eval_subset, tag="sft_eval")
        rewards  = [r["reward"] for r in records]
        exec_rate    = sum(r["executable"] for r in records) / len(records)
        correct_rate = sum(r["correct"]    for r in records) / len(records)
        mean_reward  = sum(rewards) / len(rewards)
        wandb.log({
            "sft/mean_reward":  mean_reward,
            "sft/exec_rate":    exec_rate,
            "sft/correct_rate": correct_rate,
            "sft/step":         state.global_step,
        })
        print(f"[Step {state.global_step}] reward={mean_reward:.3f} exec={exec_rate:.2f} correct={correct_rate:.2f}")

In [21]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
eval_subset = train_dataset.select(range(3))
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,   
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1,                          
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        report_to="wandb",
        output_dir = "outputs",
    ),
)

In [ ]:
trainer.add_callback(RewardLoggingCallback(model, tokenizer, eval_subset, every_n_steps=100))
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 11,913 | Num Epochs = 1 | Total steps = 1,490
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)


Step,Training Loss


In [ ]:
model.save_pretrained("outputs/stage1_sft_final")
tokenizer.save_pretrained("outputs/stage1_sft_final")
results_after, rate_after = evaluate_compilability(
    model, tokenizer, test_dataset, 
    tag="AFTER finetune",
)

print(f"\n  Before : {rate_before:.1f}%")
print(f"  After  : {rate_after:.1f}%")
print(f"  Delta  : {rate_after - rate_before:+.1f}%")